# Demo ABCDFSS — Cross-Domain Few-Shot Segmentation trên ISIC 2018 Task 1

Notebook này **chạy và minh hoạ** dự án *Adapt Before Comparison for Cross-Domain Few-Shot Segmentation* (ABCDFSS, CVPR 2024) với bộ dữ liệu
[ISIC2018 Challenge Task 1 — Data Segmentation](https://www.kaggle.com/datasets/tschandl/isic2018-challenge-task1-data-segmentation).

## Bài toán

Few-shot semantic segmentation (FSS): cho trước

| Thành phần | Ý nghĩa |
|---|---|
| **support image** | ảnh tổn thương da mẫu |
| **support mask** | mask nhị phân của tổn thương trong ảnh support |
| **query image** | ảnh cần phân đoạn |
| **query mask (GT)** | mask đáp án, chỉ dùng để đánh giá |

Mục tiêu: dự đoán mask tổn thương trên ảnh query chỉ từ **một** (hoặc vài) cặp support.

## Pipeline của repo

1. **Feature extraction** — ResNet-50 pretrained trích đặc trưng ở nhiều tầng.
2. **Task adaptation** (`core/contrastivehead.py`) — fit một *contrastive feature transformer* nhỏ (1×1 conv) cho từng lớp, cho phép đặc trưng query/support "gần nhau" hơn mà không cần train lại backbone.
3. **Comparison** (`core/denseaffinity.py`) — dense affinity matching giữa đặc trưng query và support, lọc bằng support mask.
4. **Threshold** — Otsu kết hợp `pred_mean` để ra mask nhị phân.
5. **(Tuỳ chọn) DenseCRF** — post-processing làm mượt biên.

## Dữ liệu

ISIC 2018 Task 1 gồm **2594 ảnh dermoscopic** + mask ground-truth cho tập train. Notebook có thể tải:

- **subset** (mặc định): 18 cặp ảnh + mask để demo nhanh;
- **full**: toàn bộ dataset Kaggle (~13.9 GB);
- **skip**: dùng dữ liệu bạn đã tải sẵn.

> ⚠️ **Lưu ý rất quan trọng về nhãn lớp**
>
> Repo gốc được viết cho bản `heyoujue/isic2018-classwise` — ảnh đã được chia sẵn vào 3 thư mục lớp. Bộ `tschandl/...` mà bạn yêu cầu **chỉ có ảnh Input và GroundTruth, không có nhãn chẩn đoán**. Notebook sẽ tự tạo **3 pseudo-class** bằng cách chia đều danh sách ảnh (theo thứ tự tên file, `index % 3`) để pipeline few-shot có thể chạy. Cách chia này **chỉ phục vụ demo**; kết quả không dùng để so sánh trực tiếp với số liệu trong paper.

## Nội dung notebook

| Mục | Nội dung |
|---|---|
| 0 | Kiểm tra môi trường |
| 1 | Cấu hình |
| 2 | Tải dữ liệu Kaggle |
| 3 | Adapter: tạo layout class-wise mà repo yêu cầu |
| 4 | Load dataset + xem một episode |
| 5 | Khởi tạo model + inference một episode |
| 6 | Đánh giá trên N episode + biểu đồ |
| 7 | (Tuỳ chọn) DenseCRF post-processing |
| 8 | Chạy toàn bộ benchmark bằng CLI |
| 9 | Nhận xét & lưu ý |

## 0. Kiểm tra môi trường

Chạy cell dưới để kiểm tra các thư viện cần thiết. Nếu thiếu, bỏ dấu `#` ở dòng `%pip install` rồi chạy lại.

**Yêu cầu phần cứng gợi ý:** GPU ≥ 4 GB VRAM (GTX 1650 chạy được). Notebook vẫn chạy được trên CPU, chỉ chậm hơn.

Nếu chưa có môi trường ảo, tạo một lần:

```bash
python3 -m venv .venv
source .venv/bin/activate        # Windows: .venv\Scripts\activate
pip install torch torchvision opencv-python numpy pandas matplotlib tensorboardX kagglehub
```

Trong VSCode, chọn interpreter/kernel trỏ tới `.venv`. Muốn dùng GPU, cài bản PyTorch CUDA phù hợp từ [pytorch.org](https://pytorch.org/get-started/locally/).

In [8]:
# Cài đặt — trên Colab cell này tự cài thư viện còn thiếu; máy local thì làm theo gợi ý.
# Nếu muốn cài tay: %pip install -q torch torchvision opencv-python numpy pandas matplotlib tensorboardX kagglehub

import importlib, os, subprocess, sys

def _in_colab() -> bool:
    """True khi kernel là runtime Google Colab."""
    if "google.colab" in sys.modules:
        return True
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return bool(os.environ.get("COLAB_RELEASE_TAG") or os.environ.get("COLAB_GPU"))

IN_COLAB = _in_colab()

# Tên module -> tên package trên PyPI (khác nhau ở cv2 / PIL).
PIP_NAMES = {"cv2": "opencv-python", "PIL": "pillow"}
REQUIRED = ["torch", "torchvision", "cv2", "numpy", "pandas", "matplotlib",
            "PIL", "tensorboardX", "kagglehub"]

missing = []
for name in REQUIRED:
    try:
        importlib.import_module(name)
    except Exception as exc:
        missing.append((name, str(exc)))

if missing:
    pkgs = [PIP_NAMES.get(n, n) for n, _ in missing]
    print("⚠ Thiếu thư viện:", ", ".join(n for n, _ in missing))
    if IN_COLAB:
        print("Colab: đang cài tự động →", " ".join(pkgs))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
        importlib.invalidate_caches()
    else:
        print("Cài bằng: %pip install -q " + " ".join(pkgs))
else:
    print("✅ Đã có đủ thư viện.")

import torch, torchvision
print("Python     :", sys.version.split()[0])
print("PyTorch    :", torch.__version__, "| torchvision:", torchvision.__version__)
print("CUDA       :", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU        :", props.name, f"({props.total_memory / 1e9:.1f} GB)")
print("Môi trường :", "Google Colab" if IN_COLAB else "local")


⚠ Thiếu thư viện: tensorboardX
Colab: đang cài tự động → tensorboardX
Python     : 3.13.18
PyTorch    : 2.14.1+cpu | torchvision: 0.29.1+cpu
CUDA       : False
Môi trường : Google Colab


## 1. Cấu hình

Các tham số quan trọng (có thể đổi trực tiếp hoặc qua biến môi trường):

| Biến / tham số | Mặc định | Ý nghĩa |
|---|---|---|
| `DOWNLOAD_MODE` | `"subset"` | `subset`, `full`, hoặc `skip` |
| `DEMO_EPISODES` | `9` | số episode đem đánh giá (nên là bội số của 3) |
| `IMG_SIZE` | `400` | kích thước ảnh đầu vào (repo dùng 400) |
| `N_SHOT` | `1` | số ảnh support (1-shot / 5-shot) |
| `ALLOW_SYNTHETIC_FALLBACK` | `True` | nếu không tải được dữ liệu, tạo dữ liệu giả để notebook vẫn chạy |
| `BTL_XLA_ROOT` | *(tự dò)* | đường dẫn tới thư mục gốc repo, dùng khi kernel khởi động ở thư mục khác |
| `BTL_XLA_REPO_URL` | repo hiện tại | repo để tự clone khi chạy trên Colab |

Cell cấu hình tự dò thư mục gốc của repo, không phụ thuộc thư mục làm việc của kernel: kiểm tra thư mục chứa notebook, các thư mục cha của cwd, rồi tìm trong phạm vi gần.

**Trên Google Colab**, VM không có sẵn mã nguồn: cell sẽ tự `git clone` repo về `/content/BTL_XLA` (đổi bằng `BTL_XLA_ROOT`, đổi repo bằng `BTL_XLA_REPO_URL`). Nếu repo private, hãy clone thủ công rồi đặt `BTL_XLA_ROOT`.

Đặt biến môi trường trước khi mở notebook, ví dụ:

```bash
DEMO_EPISODES=30 DEMO_IMG_SIZE=320 jupyter notebook demo.ipynb
```

In [9]:
import os, sys, json, shutil
from pathlib import Path

def _looks_like_project(p: Path) -> bool:
    return (p / "main.py").is_file() and (p / "core" / "runner.py").is_file()

def _running_in_colab() -> bool:
    """True khi kernel là runtime Google Colab (nơi repo chưa có sẵn)."""
    if "google.colab" in sys.modules:
        return True
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return bool(os.environ.get("COLAB_RELEASE_TAG") or os.environ.get("COLAB_GPU"))

def _clone_repo(dest: Path):
    """Clone mã nguồn repo (dùng khi chạy trên Colab)."""
    import subprocess
    url = os.environ.get("BTL_XLA_REPO_URL",
                         "https://github.com/tiudan147/BTL_XLA.git")
    print(f"⬇ Chưa có mã nguồn — đang clone:\n   {url}\n   → {dest}")
    res = subprocess.run(["git", "clone", "--depth", "1", url, str(dest)],
                         capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(
            "Clone repo thất bại:\n" + (res.stderr or res.stdout).strip() +
            "\nNếu repo là private, hãy clone thủ công rồi đặt biến môi trường "
            "BTL_XLA_ROOT trỏ tới thư mục vừa clone."
        )
    if not _looks_like_project(dest):
        raise RuntimeError(
            f"Đã clone {url} nhưng không thấy main.py + core/runner.py trong {dest}."
        )
    print("✅ Đã clone xong mã nguồn.")

def _first_project_along(start: Path):
    """Đi ngược từ `start` lên các thư mục cha, trả về repo đầu tiên tìm được."""
    try:
        start = Path(start).expanduser().resolve()
    except OSError:
        return None
    for p in [start, *start.parents]:
        if _looks_like_project(p):
            return p
    return None

def _find_nearby(root: Path, max_depth: int = 3, max_dirs: int = 5000):
    """Duyệt BFS có giới hạn quanh `root` (bỏ qua thư mục ẩn và thư mục nặng)."""
    skip = {".git", ".venv", "venv", "env", "__pycache__", "node_modules",
            "site-packages", ".cache", ".local", ".config", "datasets",
            "demo_outputs", "logs", "drive", "sample_data",
            # thư mục hệ thống: tránh quét cây quá lớn
            "usr", "proc", "sys", "dev", "boot", "lib", "lib64", "bin",
            "sbin", "snap", "var", "run", "srv", "mnt", "media", "nix"}
    queue, visited = [(Path(root), 0)], 0
    while queue and visited < max_dirs:
        cur, depth = queue.pop(0)
        visited += 1
        try:
            children = [c for c in cur.iterdir()
                        if c.is_dir() and not c.name.startswith(".") and c.name not in skip]
        except OSError:
            continue
        for child in children:
            if _looks_like_project(child):
                return child
            if depth + 1 < max_depth:
                queue.append((child, depth + 1))
    return None

def find_project_root():
    """Tìm thư mục gốc repo mà không phụ thuộc vào cwd của kernel."""
    # 1) Chỉ định thủ công: hữu ích khi kernel chạy ở thư mục khác.
    for var in ("BTL_XLA_ROOT", "PROJECT_ROOT"):
        val = os.environ.get(var)
        if val:
            found = _first_project_along(Path(val))
            if found and (found / "core" / "runner.py").is_file():
                return found

    # 2) Thư mục chứa notebook (VS Code chèn biến này), rồi tới thư mục làm việc.
    starts = []
    nb_file = globals().get("__vsc_ipynb_file__")
    if nb_file:
        starts.append(Path(nb_file).parent)
    try:
        starts.append(Path.cwd())
    except OSError:
        pass

    for start in starts:
        found = _first_project_along(start)
        if found:
            return found

    # 3) Tìm quanh các vị trí trên (tối đa 3 cấp), sau đó quanh thư mục home.
    search_roots = list(starts)
    try:
        home = Path.home()
        if home not in search_roots:
            search_roots.append(home)
    except (OSError, RuntimeError):
        pass

    for start in search_roots:
        found = _find_nearby(start)
        if found:
            return found

    # 4) Colab: VM không có sẵn mã nguồn → clone repo về /content/BTL_XLA.
    if _running_in_colab():
        dest = Path(os.environ.get("BTL_XLA_ROOT", "/content/BTL_XLA")).expanduser()
        if _looks_like_project(dest):
            return dest
        if dest.exists():
            print(f"⚠ {dest} đã tồn tại nhưng không phải repo — không tự clone. "
                  "Hãy kiểm tra thư mục này.")
        else:
            _clone_repo(dest)
            if _looks_like_project(dest):
                return dest
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None or not (PROJECT_ROOT / "core" / "runner.py").is_file():
    try:
        cwd = Path.cwd()
    except OSError:
        cwd = "<không xác định>"
    raise RuntimeError(
        "Không tìm thấy mã nguồn dự án (main.py + core/runner.py).\n"
        f"  Thư mục làm việc hiện tại: {cwd}\n"
        "Cách xử lý:\n"
        "  • Mở notebook ngay trong thư mục gốc của repo, hoặc\n"
        "  • Đặt biến môi trường BTL_XLA_ROOT=/đường/dẫn/tới/BTL_XLA rồi khởi động lại kernel,\n"
        "    hoặc sửa trực tiếp biến PROJECT_ROOT trong cell này.\n"
        "  • Trên Colab: cell này tự clone repo (đặt BTL_XLA_REPO_URL nếu dùng repo khác)."
    )

# Làm việc từ repo root để mọi đường dẫn tương đối (logs, outputs, ...) nhất quán.
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

# ---- Đường dẫn -------------------------------------------------------------
DATASET_HANDLE = "tschandl/isic2018-challenge-task1-data-segmentation"
DATASETS_DIR   = PROJECT_ROOT / "datasets"
RAW_DIR        = DATASETS_DIR / "isic2018_raw"
CLASSWISE_DIR  = DATASETS_DIR / "isic2018_classwise_demo"
# Nếu đã có bản class-wise của tác giả (heyoujue/isic2018-classwise):
# CLASSWISE_DIR = DATASETS_DIR / "isic2018-classwise"
OUTPUT_DIR     = PROJECT_ROOT / "demo_outputs"

# ---- Tham số demo ----------------------------------------------------------
DOWNLOAD_MODE = os.environ.get("DEMO_DOWNLOAD_MODE", "subset").lower()   # subset | full | skip
DEMO_EPISODES = int(os.environ.get("DEMO_EPISODES", "9"))
IMG_SIZE      = int(os.environ.get("DEMO_IMG_SIZE", "400"))
N_SHOT        = int(os.environ.get("DEMO_NSHOT", "1"))
SEED          = int(os.environ.get("DEMO_SEED", "1234"))
VERBOSITY     = int(os.environ.get("DEMO_VERBOSITY", "0"))
POSTPROCESSING = os.environ.get("DEMO_POSTPROCESSING", "off")             # off | always | dynamic
ADAPT_EVERY_EPISODE = os.environ.get("DEMO_ADAPT_EVERY_EPISODE", "0") == "1"
ALLOW_SYNTHETIC_FALLBACK = os.environ.get("DEMO_ALLOW_SYNTHETIC", "1") == "1"

# 18 ID trải đều trong tập train. Mỗi ID có 1 ảnh .jpg và 1 mask _segmentation.png.
SUBSET_IDS = [
    "ISIC_0000000", "ISIC_0000169", "ISIC_0000354", "ISIC_0000882",
    "ISIC_0008403", "ISIC_0010028", "ISIC_0010379", "ISIC_0011161",
    "ISIC_0012227", "ISIC_0012685", "ISIC_0013120", "ISIC_0013490",
    "ISIC_0013862", "ISIC_0014353", "ISIC_0014748", "ISIC_0015007",
    "ISIC_0015419", "ISIC_0016072",
]

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATASETS_DIR :", DATASETS_DIR)
print("DOWNLOAD_MODE:", DOWNLOAD_MODE)
print("DEMO_EPISODES:", DEMO_EPISODES, "| IMG_SIZE:", IMG_SIZE, "| N_SHOT:", N_SHOT)

⬇ Chưa có mã nguồn — đang clone:
   https://github.com/tiudan147/BTL_XLA.git
   → /content/BTL_XLA
✅ Đã clone xong mã nguồn.
PROJECT_ROOT : /content/BTL_XLA
DATASETS_DIR : /content/BTL_XLA/datasets
DOWNLOAD_MODE: subset
DEMO_EPISODES: 9 | IMG_SIZE: 400 | N_SHOT: 1


## 2. Tải dữ liệu ISIC 2018 Task 1 từ Kaggle

### Xác thực Kaggle

Kaggle thường yêu cầu token. Trong nhiều trường hợp, `kagglehub` vẫn tải được dataset public mà không cần token; nếu gặp lỗi 401/403, làm theo các bước:

1. Đăng nhập [kaggle.com](https://www.kaggle.com) → **Settings** → **API** → **Create New Token**.
2. Lưu file `kaggle.json` vào `~/.kaggle/`:

```bash
mkdir -p ~/.kaggle
mv ~/Downloads/kaggle.json ~/.kaggle/kaggle.json
chmod 600 ~/.kaggle/kaggle.json
```

3. Hoặc đặt biến môi trường `KAGGLE_USERNAME` và `KAGGLE_KEY`.
4. Cài `kagglehub`: `%pip install -q -U kagglehub`.

### Ba chế độ tải

- **`subset`** (mặc định): tải 18 cặp ảnh + mask (~vài chục MB) — đủ để demo và kiểm tra pipeline.
- **`full`**: tải toàn bộ ~13.9 GB (2594 ảnh train + mask, 1000 ảnh test). Cần nhiều dung lượng và thời gian.
- **`skip`**: dùng dữ liệu đã có trong `datasets/isic2018_raw/` (đúng cấu trúc thư mục của Kaggle).

Nếu việc tải thất bại (thiếu token hoặc mất mạng), notebook sẽ báo lỗi và (nếu `ALLOW_SYNTHETIC_FALLBACK=True`) chuyển sang dữ liệu giả ở mục 3 để bạn vẫn xem được toàn bộ luồng chạy.

In [ ]:
DATA_SOURCE = "missing"
RAW_ROOT = None

def _isic_root_of(base):
    base = Path(base)
    if not base.exists():
        return None
    if (base / "ISIC2018_Task1-2_Training_Input").is_dir():
        return base
    for p in base.iterdir():
        if p.is_dir() and (p / "ISIC2018_Task1-2_Training_Input").is_dir():
            return p
    for p in base.rglob("ISIC2018_Task1-2_Training_Input"):
        if p.is_dir():
            return p.parent
    return None

def _download_subset():
    import kagglehub
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    todo = []
    for img_id in SUBSET_IDS:
        todo.append(f"ISIC2018_Task1-2_Training_Input/{img_id}.jpg")
        todo.append(f"ISIC2018_Task1_Training_GroundTruth/{img_id}_segmentation.png")

    for idx, rel in enumerate(todo, 1):
        dst = RAW_DIR / rel
        if dst.exists():
            print(f"[{idx:02d}/{len(todo)}] có sẵn   {rel}")
            continue
        out = kagglehub.dataset_download(DATASET_HANDLE, path=rel, output_dir=str(RAW_DIR))
        found = Path(out)
        if found.is_dir():
            hits = list(found.rglob(Path(rel).name)) or list(RAW_DIR.rglob(Path(rel).name))
            found = hits[0] if hits else found
        if not found.is_file():
            hits = list(RAW_DIR.rglob(Path(rel).name))
            if not hits:
                raise FileNotFoundError(rel)
            found = hits[0]
        if found.resolve() != dst.resolve():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(found, dst)
        print(f"[{idx:02d}/{len(todo)}] OK        {rel}")
    return RAW_DIR

def _kaggle_auth_hint():
    print('''
Không tải được dữ liệu từ Kaggle. Cần xác thực (miễn phí):
  1. kaggle.com → Settings → API → "Create New Token" (tải kaggle.json)
  2. mkdir -p ~/.kaggle && mv ~/Downloads/kaggle.json ~/.kaggle/
     chmod 600 ~/.kaggle/kaggle.json
  3. Hoặc export KAGGLE_USERNAME=... KAGGLE_KEY=...
Sau đó chạy lại cell này. Nếu đã có dữ liệu, đặt DOWNLOAD_MODE='skip'.
''')

if DOWNLOAD_MODE == "skip":
    print("DOWNLOAD_MODE='skip' → không tải dữ liệu.")
elif DOWNLOAD_MODE == "full":
    try:
        import kagglehub
        print("Đang tải TOÀN BỘ dataset (~13.9 GB, có thể mất khá lâu)...")
        RAW_ROOT = Path(kagglehub.dataset_download(DATASET_HANDLE))
    except Exception as exc:
        print("Lỗi khi tải:", repr(exc))
        _kaggle_auth_hint()
elif DOWNLOAD_MODE == "subset":
    try:
        RAW_ROOT = _download_subset()
    except Exception as exc:
        print("Lỗi khi tải:", repr(exc))
        _kaggle_auth_hint()
else:
    raise ValueError("DOWNLOAD_MODE phải là 'subset', 'full' hoặc 'skip'")

if RAW_ROOT is not None:
    RAW_ROOT = _isic_root_of(RAW_ROOT)
if RAW_ROOT is None:
    RAW_ROOT = _isic_root_of(RAW_DIR)

if RAW_ROOT is not None:
    raw_root = Path(RAW_ROOT)
    img_dir = raw_root / "ISIC2018_Task1-2_Training_Input"
    gt_dir  = raw_root / "ISIC2018_Task1_Training_GroundTruth"
    n_img = len(list(img_dir.rglob("*.jpg"))) if img_dir.is_dir() else 0
    n_gt  = len(list(gt_dir.rglob("*_segmentation.png"))) if gt_dir.is_dir() else 0
    if n_img > 0 and n_gt > 0:
        DATA_SOURCE = "real"
        print(f"✅ Dữ liệu thật: {raw_root}")
        print(f"   {n_img} ảnh input | {n_gt} mask ground-truth")
    else:
        RAW_ROOT = None
        print("⚠ Không tìm thấy ảnh/mask phù hợp trong", raw_root)
else:
    print("ℹ Chưa có dữ liệu thật trong", RAW_DIR)

print("DATA_SOURCE =", DATA_SOURCE)

100%|██████████| 48.8k/48.8k [00:00<00:00, 23.9MB/s]

[01/36] OK        ISIC2018_Task1-2_Training_Input/ISIC_0000000.jpg


Using Colab cache for faster access to the 'isic2018-challenge-task1-data-segmentation' dataset.
[02/36] OK        ISIC2018_Task1_Training_GroundTruth/ISIC_0000000_segmentation.png


## 3. Adapter: tạo layout class-wise

`data/isic.py` (của repo) mong đợi cấu trúc:

```text
<DATAPATH>/
├── ISIC2018_Task1-2_Training_Input/
│   ├── 1/<ISIC_id>.jpg
│   ├── 2/<ISIC_id>.jpg
│   └── 3/<ISIC_id>.jpg
└── ISIC2018_Task1_Training_GroundTruth/
    └── <ISIC_id>_segmentation.png
```

Vì bộ `tschandl/...` không có thư mục `1/ 2/ 3/`, cell dưới sẽ:

1. Gom các cặp `ảnh ↔ mask` khớp tên.
2. Sắp xếp theo `ISIC_id` rồi gán `pseudo_class = i % 3 + 1` (chia đều, tất định).
3. Tạo symlink (hoặc copy nếu không tạo được symlink) vào `datasets/isic2018_classwise_demo/`.

> Nếu bạn đã có bản `heyoujue/isic2018-classwise`, đặt `CLASSWISE_DIR` trỏ tới thư mục đó — notebook sẽ dùng trực tiếp.
>
> Nếu không có dữ liệu thật, cell sẽ tạo **dữ liệu giả lập** (ảnh nền màu da + hình ellipse ngẫu nhiên + mask) để minh hoạ pipeline. Dữ liệu giả lập được đánh dấu rõ và **không dùng để đánh giá**.

In [ ]:
import numpy as np

def has_classwise_layout(root) -> bool:
    root = Path(root)
    base = root / "ISIC2018_Task1-2_Training_Input"
    if not base.is_dir():
        return False
    for cat in ("1", "2", "3"):
        if not (base / cat).is_dir() or not any((base / cat).glob("*.jpg")):
            return False
    return True

def collect_pairs(raw_root):
    raw_root = Path(raw_root)
    img_dir = raw_root / "ISIC2018_Task1-2_Training_Input"
    gt_dir  = raw_root / "ISIC2018_Task1_Training_GroundTruth"
    imgs = {p.stem: p for p in img_dir.glob("*.jpg")}
    gts  = {p.name.replace("_segmentation.png", ""): p
            for p in gt_dir.glob("*_segmentation.png")} if gt_dir.is_dir() else {}
    ids = sorted(set(imgs) & set(gts))
    return ids, imgs, gts

def make_synthetic_raw(out_dir, n_images=18, size=512, seed=SEED):
    from PIL import Image, ImageDraw, ImageFilter
    rng = np.random.default_rng(seed)
    out_dir = Path(out_dir)
    img_dir = out_dir / "ISIC2018_Task1-2_Training_Input"
    gt_dir  = out_dir / "ISIC2018_Task1_Training_GroundTruth"
    img_dir.mkdir(parents=True, exist_ok=True)
    gt_dir.mkdir(parents=True, exist_ok=True)

    for i in range(n_images):
        img_id = f"SYN_{i:05d}"
        if (img_dir / f"{img_id}.jpg").exists() and (gt_dir / f"{img_id}_segmentation.png").exists():
            continue
        H = W = size
        bg = np.zeros((H, W, 3), np.uint8)
        bg[..., 0] = rng.integers(170, 225)
        bg[..., 1] = rng.integers(120, 175)
        bg[..., 2] = rng.integers(95, 145)
        bg = np.clip(bg + rng.normal(0, 10, (H, W, 3)), 0, 255).astype(np.uint8)
        img = Image.fromarray(bg)

        cx, cy = (int(v) for v in rng.integers(size // 3, 2 * size // 3, size=2))
        rx, ry = (int(v) for v in rng.integers(size // 8, size // 3, size=2))
        color = tuple(int(v) for v in rng.integers(45, 165, size=3))
        ImageDraw.Draw(img).ellipse([cx - rx, cy - ry, cx + rx, cy + ry], fill=color)
        img = img.filter(ImageFilter.GaussianBlur(1.2))
        img.save(img_dir / f"{img_id}.jpg", quality=92)

        mask = Image.new("L", (W, H), 0)
        ImageDraw.Draw(mask).ellipse([cx - rx, cy - ry, cx + rx, cy + ry], fill=255)
        mask.save(gt_dir / f"{img_id}_segmentation.png")
    return out_dir

def build_classwise(raw_root, out_dir, ids, imgs, gts):
    out_dir = Path(out_dir)
    marker = out_dir / ".demo_adapter.json"

    if has_classwise_layout(out_dir) and not marker.exists():
        print("ℹ Dùng layout class-wise có sẵn:", out_dir)
        return out_dir

    if out_dir.exists():
        if marker.exists():
            shutil.rmtree(out_dir)
        else:
            raise RuntimeError(
                f"{out_dir} đã tồn tại nhưng không phải layout do notebook tạo. "
                "Hãy đổi CLASSWISE_DIR hoặc xoá thư mục đó thủ công."
            )

    train_in = out_dir / "ISIC2018_Task1-2_Training_Input"
    gt_out   = out_dir / "ISIC2018_Task1_Training_GroundTruth"
    for cat in ("1", "2", "3"):
        (train_in / cat).mkdir(parents=True, exist_ok=True)
    gt_out.mkdir(parents=True, exist_ok=True)

    counts = {"1": 0, "2": 0, "3": 0}
    for i, img_id in enumerate(ids):
        cls = str(i % 3 + 1)
        for src, dst in [
            (imgs[img_id], train_in / cls / f"{img_id}.jpg"),
            (gts[img_id],  gt_out / f"{img_id}_segmentation.png"),
        ]:
            try:
                os.symlink(Path(src).resolve(), dst)
            except OSError:
                shutil.copy2(src, dst)
        counts[cls] += 1

    marker.write_text(json.dumps({
        "source": str(Path(raw_root)), "n_ids": len(ids),
        "counts": counts, "seed": SEED,
        "note": "Pseudo-classes built by demo notebook (index % 3).",
    }, ensure_ascii=False, indent=2))
    print("✅ Đã tạo layout class-wise:", out_dir)
    print("   Số ảnh mỗi pseudo-class:", counts)
    return out_dir

# ---- Chọn nguồn dữ liệu ----------------------------------------------------
if DATA_SOURCE == "real":
    if has_classwise_layout(RAW_ROOT):
        print("ℹ RAW_ROOT đã có layout class-wise → dùng trực tiếp.")
        DATAPATH = Path(RAW_ROOT)
    else:
        ids, imgs, gts = collect_pairs(RAW_ROOT)
        print("Số cặp ảnh/mask tìm thấy:", len(ids))
        if len(ids) < 3:
            raise RuntimeError("Cần ít nhất 3 cặp ảnh/mask để tạo 3 lớp.")
        DATAPATH = build_classwise(RAW_ROOT, CLASSWISE_DIR, ids, imgs, gts)

elif ALLOW_SYNTHETIC_FALLBACK:
    print("⚠ KHÔNG CÓ DỮ LIỆU THẬT → tạo dữ liệu giả lập để notebook vẫn chạy được.")
    print("   Dữ liệu giả lập CHỈ để kiểm tra pipeline, KHÔNG dùng để đánh giá mô hình.")
    SYN_RAW = make_synthetic_raw(
        DATASETS_DIR / "isic2018_synthetic_raw",
        n_images=max(18, DEMO_EPISODES + 9), size=512, seed=SEED,
    )
    ids, imgs, gts = collect_pairs(SYN_RAW)
    DATAPATH = build_classwise(SYN_RAW, CLASSWISE_DIR, ids, imgs, gts)
    DATA_SOURCE = "synthetic"

else:
    raise RuntimeError(
        "Không có dữ liệu thật và ALLOW_SYNTHETIC_FALLBACK=False. "
        "Hãy cấu hình Kaggle token hoặc tải dữ liệu vào datasets/isic2018_raw/."
    )

print(f"\nDATA_SOURCE = {DATA_SOURCE}")
print(f"DATAPATH    = {DATAPATH}")

## 4. Load dataset và xem một episode

`FSSDataset` là lớp builder của repo. Mỗi lần lấy `__getitem__`, `DatasetISIC` sẽ:

1. Chọn lớp theo `idx % 3` (0 → class 1, 1 → class 2, 2 → class 3).
2. Random một ảnh query và một ảnh support **khác query** trong cùng lớp.
3. Resize về `IMG_SIZE × IMG_SIZE`, chuẩn hoá ImageNet, trả về dict các tensor.

Cell dưới in shape của batch để bạn đối chiếu.

In [ ]:
import warnings
import numpy as np
import torch
from data.dataset import FSSDataset
from utils import commonutils

# Cảnh báo pin_memory chỉ xuất hiện khi chạy CPU, không ảnh hưởng kết quả.
warnings.filterwarnings("ignore", message="'pin_memory' argument is set as true")

commonutils.fix_randseed(SEED)

FSSDataset.initialize(img_size=IMG_SIZE, datapath=str(DATAPATH))
loader = FSSDataset.build_dataloader(
    benchmark="isic", bsz=1, nworker=0, fold=0, split="test", shot=N_SHOT
)
print("Số ảnh trong dataset:", len(loader.dataset))

batch = next(iter(loader))
print("class_id:", int(batch["class_id"].item()))
for key in ("query_img", "query_mask", "support_imgs", "support_masks"):
    print(f"{key:14s}: {tuple(batch[key].shape)}")
print("query_name    :", batch["query_name"][0])
print("support_names :", batch["support_names"][0])

### Hiển thị episode

Các hàm dưới chuẩn hoá ảnh về `[0, 1]`, hiển thị mask và overlay dự đoán. Màu sắc lấy từ palette trung tính, tự thích ứng nền sáng/tối của notebook.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb, LinearSegmentedColormap

def _theme():
    try:
        dark = sum(to_rgb(plt.rcParams["figure.facecolor"])) / 3 < 0.5
    except Exception:
        dark = False
    if dark:
        return dict(surface="#1a1a19", ink="#f0efec", ink2="#c3c2b7",
                    grid="#2c2c2a", axis="#383835", series="#3987e5", accent="#e66767")
    return dict(surface="#fcfcfb", ink="#0b0b0b", ink2="#52514e",
                grid="#e1e0d9", axis="#c3c2b7", series="#2a78d6", accent="#e34948")

THEME = _theme()
if THEME["surface"] == "#1a1a19":
    PROB_CMAP = LinearSegmentedColormap.from_list(
        "prob_dark", ["#1a1a19", "#184f95", "#3987e5", "#86b6ef"])
else:
    PROB_CMAP = LinearSegmentedColormap.from_list(
        "prob_light", ["#fcfcfb", "#86b6ef", "#2a78d6", "#0d366b"])

IMG_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMG_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def denorm_img(t):
    t = t.detach().cpu().float()
    if t.ndim == 4:
        t = t[0]
    t = t.permute(1, 2, 0).numpy()
    return np.clip(t * IMG_STD + IMG_MEAN, 0, 1)

def as_mask(t):
    t = t.detach().cpu().float()
    while t.ndim > 2:
        t = t[0]
    return t.numpy()

def _style_ax(ax, title=""):
    ax.set_title(title, fontsize=9, color=THEME["ink"], pad=4)
    ax.axis("off")

def plot_inputs(support_imgs, support_masks, query_img, query_mask, suptitle=""):
    k = support_imgs.shape[0]
    ncols = 2 * k + 2
    fig, axes = plt.subplots(1, ncols, figsize=(2.7 * ncols, 2.9))
    axes = np.atleast_1d(axes)
    c = 0
    for i in range(k):
        axes[c].imshow(denorm_img(support_imgs[i]));      _style_ax(axes[c], f"Support {i+1}: ảnh");  c += 1
        axes[c].imshow(as_mask(support_masks[i]), cmap="gray", vmin=0, vmax=1)
        _style_ax(axes[c], f"Support {i+1}: mask");       c += 1
    axes[c].imshow(denorm_img(query_img));                _style_ax(axes[c], "Query: ảnh");          c += 1
    axes[c].imshow(as_mask(query_mask), cmap="gray", vmin=0, vmax=1)
    _style_ax(axes[c], "Query: ground-truth")
    if suptitle:
        fig.suptitle(suptitle, fontsize=10, color=THEME["ink"])
    fig.tight_layout()
    plt.show()

def overlay_mask(img, mask, color=None, alpha=0.35):
    color = np.array(to_rgb(color or THEME["accent"]))
    out = img.copy()
    m = mask > 0.5
    out[m] = (1 - alpha) * out[m] + alpha * color
    return np.clip(out, 0, 1)

def _result_panels(support_img, support_mask, query_img, query_gt, coarse, pred):
    return [
        (denorm_img(support_img), "Support: ảnh",             None,      None),
        (as_mask(support_mask),   "Support: mask",            "gray",    (0, 1)),
        (denorm_img(query_img),   "Query: ảnh",               None,      None),
        (as_mask(query_gt),       "Query: GT",                "gray",    (0, 1)),
        (as_mask(coarse),         "Coarse probability",       PROB_CMAP, (0, 1)),
        (overlay_mask(denorm_img(query_img), as_mask(pred)),
                                  "Dự đoán (overlay)",        None,      None),
    ]

def plot_result(support_img, support_mask, query_img, query_gt, coarse, pred, title=""):
    fig, axes = plt.subplots(1, 6, figsize=(16, 2.9))
    for ax, (im, t, cm, vr) in zip(axes, _result_panels(
            support_img, support_mask, query_img, query_gt, coarse, pred)):
        if cm:
            ax.imshow(im, cmap=cm, vmin=vr[0], vmax=vr[1])
        else:
            ax.imshow(im)
        _style_ax(ax, t)
    if title:
        fig.suptitle(title, fontsize=10, color=THEME["ink"])
    fig.tight_layout()
    plt.show()

def plot_records_grid(records, n=4, title=""):
    recs = records[:n]
    fig, axes = plt.subplots(len(recs), 6, figsize=(16, 2.7 * len(recs)), squeeze=False)
    for row, rec in enumerate(recs):
        panels = _result_panels(rec["support_img"], rec["support_mask"],
                                rec["query_img"], rec["query_gt"],
                                rec["coarse"], rec["pred"])
        for col, (ax, (im, t, cm, vr)) in enumerate(zip(axes[row], panels)):
            if cm:
                ax.imshow(im, cmap=cm, vmin=vr[0], vmax=vr[1])
            else:
                ax.imshow(im)
            label = t if col else f"#{rec['step']} · lớp {rec['class_id']+1} · IoU {rec['iou']:.1f}%"
            _style_ax(ax, label)
    if title:
        fig.suptitle(title, fontsize=10, color=THEME["ink"])
    fig.tight_layout()
    plt.show()
    return fig

plot_inputs(
    batch["support_imgs"][0], batch["support_masks"][0],
    batch["query_img"][0], batch["query_mask"][0],
    suptitle=f"Episode mẫu — pseudo-class {int(batch['class_id'].item()) + 1}",
)

## 5. Khởi tạo model và inference một episode

Phần này tái sử dụng **đúng các lớp của repo**:

- `core.backbone.Backbone` — ResNet-50 (pretrained ImageNet).
- `core.contrastivehead.FeatureMaker` — quản lý contrastive transformer theo từng lớp.
- `core.denseaffinity.DAMatComparison` — dense affinity matching.
- `core.runner.SingleSampleEval` — gom toàn bộ pipeline cho một task.

**Quick-infer (mặc định):** transformer chỉ được fit ở episode đầu tiên của mỗi lớp rồi tái sử dụng. Đặt `ADAPT_EVERY_EPISODE=True` (hoặc `--adapt-to every-episode`) để fit lại mỗi episode — chậm hơn nhiều nhưng bám sát paper hơn.

⏱ Lần chạy đầu tiên sẽ tải trọng số ResNet-50 (~100 MB) từ Internet.

In [ ]:
import argparse, time
import torch
from core import runner

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", device)

args = argparse.Namespace(
    benchmark="isic",
    datapath=str(DATAPATH),
    nshot=N_SHOT,
    adapt_to="every-episode" if ADAPT_EVERY_EPISODE else "first-episode",
    postprocessing=POSTPROCESSING,
    logpath="logs_demo",
    verbosity=VERBOSITY,
)
runner.set_args(args)
config = runner.makeConfig()
config.featext.fit_every_episode = ADAPT_EVERY_EPISODE

commonutils.fix_randseed(SEED)
feat_maker = runner.makeFeatureMaker(loader.dataset, config, device=device)

print("Backbone        :", runner.args.backbone)
print("out_channels    :", config.model.out_channels)
print("fit_every_episode:", config.featext.fit_every_episode)
print("postprocessing  :", POSTPROCESSING)

In [ ]:
sseval = runner.SingleSampleEval(batch, feat_maker)
sseval.post_proc_method = POSTPROCESSING

t0 = time.time()
sseval.forward()
forward_time = time.time() - t0

iou_single = sseval.calc_metrics()
print(f"Thời gian forward/xử lý 1 episode: {forward_time:.1f}s")
print(f"Foreground IoU của episode này  : {float(iou_single) * 100:.2f}%")
print(f"Threshold (Otsu + pred_mean)     : {float(sseval.thresh.item()):.4f}")
print(f"Tỉ lệ foreground dự đoán / GT    : {float(sseval.fgratio_pred)*100:.1f}% / {float(sseval.fgratio_gt)*100:.1f}%")

plot_result(
    sseval.s_img[0, 0].cpu(), sseval.s_mask[0, 0].cpu(),
    sseval.q_img[0].cpu(), batch["query_mask"][0].cpu(),
    sseval.logit_mask[0].cpu(), sseval.pred_mask[0].cpu(),
    title=f"Kết quả 1 episode — IoU = {float(iou_single)*100:.1f}%",
)

## 6. Đánh giá trên nhiều episode

Mỗi episode được lấy ngẫu nhiên theo seed cố định. Ta dùng `AverageMeterWrapper` của repo để tích luỹ intersection/union theo từng lớp, sau đó tính:

- **mIoU** = trung bình IoU foreground trên 3 lớp.
- **FB-IoU** = IoU foreground/background gộp toàn bộ episode.

> Với dữ liệu thật, mỗi lần chạy có thể lệch chút ít do episode được sample ngẫu nhiên. Tăng `DEMO_EPISODES` để ổn định hơn.

In [ ]:
import itertools
import pandas as pd

records = []
meter = runner.AverageMeterWrapper(loader, device, initlogger=False)
commonutils.fix_randseed(SEED)

t_start = time.time()
for step, b in enumerate(itertools.islice(loader, DEMO_EPISODES)):
    ss = runner.SingleSampleEval(b, feat_maker)
    ss.post_proc_method = POSTPROCESSING
    ss.forward()
    iou_fg = ss.calc_metrics()
    meter.update(ss)

    records.append({
        "step": step + 1,
        "class_id": int(ss.class_id),
        "iou": float(iou_fg) * 100,
        "support_img": ss.s_img[0, 0].detach().cpu(),
        "support_mask": ss.s_mask[0, 0].detach().cpu(),
        "query_img": ss.q_img[0].detach().cpu(),
        "query_gt": b["query_mask"][0].detach().cpu(),
        "coarse": ss.logit_mask[0].detach().cpu(),
        "pred": ss.pred_mask[0].detach().cpu(),
    })
    print(f"[{step+1:02d}/{DEMO_EPISODES}] lớp {int(ss.class_id)+1} | IoU = {float(iou_fg)*100:6.2f}%")

elapsed = time.time() - t_start
miou, fb_iou = meter.average_meter.compute_iou()

inter = meter.average_meter.intersection_buf[1].detach().cpu()
union = meter.average_meter.union_buf[1].detach().cpu()
per_class_iou = np.round((inter / union.clamp_min(1e-8) * 100).numpy().astype(float), 2)

counts = np.bincount([r["class_id"] for r in records], minlength=3)
summary = pd.DataFrame({
    "Pseudo-class": ["1", "2", "3"],
    "Số episode": counts.astype(int),
    "IoU (%)": per_class_iou.round(2),
})

print("\n" + "=" * 64)
print(f"mIoU  (mean IoU foreground) : {float(miou):6.2f}%")
print(f"FB-IoU (foreground-background): {float(fb_iou):6.2f}%")
print(f"Tổng thời gian               : {elapsed:.1f}s  ({elapsed/max(1, len(records)):.2f}s/episode)")
print("=" * 64)
print(summary.to_string(index=False))

### Biểu đồ IoU theo lớp

Một series duy nhất nên không cần legend; giá trị được ghi trực tiếp trên cột. Đường nét đứt là mIoU.

In [ ]:
fig_iou, ax = plt.subplots(figsize=(6.2, 3.5))
fig_iou.patch.set_facecolor(THEME["surface"])
ax.set_facecolor(THEME["surface"])

x = np.arange(len(summary))
vals = summary["IoU (%)"].to_numpy()
bars = ax.bar(x, vals, width=0.55, color=THEME["series"], edgecolor="none", zorder=3)

miou_val = float(miou)
ax.axhline(miou_val, color=THEME["ink2"], linestyle="--", linewidth=1.2, zorder=2)
ax.text(-0.45, miou_val + 1.8, f"mIoU {miou_val:.1f}%",
        ha="left", va="bottom", fontsize=8, color=THEME["ink2"])

for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 1.5, f"{val:.1f}",
            ha="center", va="bottom", fontsize=9, color=THEME["ink"])

ax.set_xticks(x, summary["Pseudo-class"])
ax.set_xlabel("Pseudo-class", color=THEME["ink2"], fontsize=9)
ax.set_ylabel("IoU (%)", color=THEME["ink2"], fontsize=9)
ax.set_ylim(0, 100)
ax.set_title("IoU theo lớp", loc="left", fontsize=11, fontweight="bold", color=THEME["ink"])
ax.grid(axis="y", color=THEME["grid"], linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color(THEME["axis"])
ax.tick_params(colors=THEME["ink2"], labelsize=9)
fig_iou.tight_layout()
plt.show()

### Kết quả định tính

Bốn episode đầu tiên trong vòng đánh giá: ảnh support, mask support, query, ground-truth, coarse probability và overlay dự đoán.

In [ ]:
fig_grid = plot_records_grid(
    records, n=min(4, len(records)),
    title="Kết quả định tính trên các episode đã đánh giá",
)

### Lưu kết quả

Lưu bảng điểm và hình vào `demo_outputs/` để đưa vào báo cáo.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUTPUT_DIR / "per_class_iou.csv", index=False)
pd.DataFrame([{"step": r["step"], "class_id": r["class_id"] + 1, "iou_percent": r["iou"]}
              for r in records]).to_csv(OUTPUT_DIR / "episodes.csv", index=False)
fig_iou.savefig(OUTPUT_DIR / "iou_per_class.png", dpi=160, bbox_inches="tight",
                facecolor=fig_iou.get_facecolor())
if "fig_grid" in globals():
    fig_grid.savefig(OUTPUT_DIR / "qualitative_grid.png", dpi=140, bbox_inches="tight",
                     facecolor=fig_grid.get_facecolor())
print("Đã lưu:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print("  ", p)

## 7. (Tuỳ chọn) DenseCRF post-processing

Repo hỗ trợ 3 chế độ: `off` (mặc định), `always`, `dynamic` (tự quyết định có refine hay không dựa trên pseudo-episode). Post-processing dùng `pydensecrf`, cần biên dịch Cython nên không cài sẵn:

```bash
pip install git+https://github.com/lucasb-eyer/pydensecrf.git
```

Sau khi cài, đặt `RUN_CRF = True` và chạy cell dưới. Lưu ý CRF chạy trên CPU nên khá chậm; paper dùng `dynamic` để tránh refine khi không có lợi.

In [ ]:
RUN_CRF = False   # đổi thành True sau khi cài pydensecrf

if RUN_CRF:
    try:
        import pydensecrf  # noqa: F401
    except ImportError as exc:
        print("Chưa cài pydensecrf:", exc)
    else:
        ss_crf = runner.SingleSampleEval(batch, feat_maker)
        ss_crf.post_proc_method = "always"
        t0 = time.time()
        ss_crf.forward()
        iou_crf = ss_crf.calc_metrics()
        print(f"CRF xong trong {time.time()-t0:.1f}s | IoU = {float(iou_crf)*100:.2f}%")
        plot_result(
            ss_crf.s_img[0, 0].cpu(), ss_crf.s_mask[0, 0].cpu(),
            ss_crf.q_img[0].cpu(), batch["query_mask"][0].cpu(),
            ss_crf.logit_mask[0].cpu(), ss_crf.pred_mask[0].cpu(),
            title=f"Sau DenseCRF — IoU = {float(iou_crf)*100:.1f}%",
        )
else:
    print("Bỏ qua CRF. Đặt RUN_CRF=True để thử sau khi cài pydensecrf.")

## 8. Chạy toàn bộ benchmark bằng CLI

Notebook ở trên chỉ chạy một số episode để demo. Muốn chạy **toàn bộ** dataset (2594 episode với bản full), dùng `main.py`:

```bash
# Quick-infer + không post-processing (mặc định, nhanh nhất)
python main.py \
  --benchmark isic \
  --datapath ./datasets/isic2018_classwise_demo \
  --nshot 1 \
  --verbosity 1

# Bám sát paper hơn: fit lại transformer cho từng episode + CRF động
python main.py \
  --benchmark isic \
  --datapath ./datasets/isic2018_classwise_demo \
  --nshot 1 \
  --adapt-to every-episode \
  --postprocessing dynamic \
  --verbosity 1
```

Kết quả in ra cuối chương trình:

```text
Result m|FB: (tensor(...), tensor(...))
```

- `m` = mIoU foreground trung bình trên 3 lớp.
- `FB` = foreground-background IoU.
- Log và TensorBoard được ghi vào `logs/` (cần `tensorboardX`).

**Lưu ý về tốc độ:** `--adapt-to every-episode` fit 25 epoch cho mỗi episode nên chậm hơn khoảng vài chục lần. Với 2594 episode, hãy cân nhắc chạy trên GPU và/hoặc lưu kết quả trung gian.

## 9. Nhận xét & lưu ý

### Notebook đã minh hoạ gì?

1. Tải và chuẩn hoá dữ liệu ISIC 2018 về đúng layout mà repo yêu cầu.
2. Sinh episode few-shot (support + query) đúng như lúc evaluate.
3. Chạy đầy đủ pipeline: ResNet-50 → task adaptation → dense affinity → threshold → mask.
4. Tính mIoU / FB-IoU và trực quan hoá kết quả.

### Những điểm cần lưu ý

- **Pseudo-class:** bộ `tschandl/...` không có nhãn chẩn đoán. Notebook chia ảnh thành 3 nhóm theo `index % 3`. Vì vậy kết quả **không so sánh trực tiếp** với bảng số trong paper (paper dùng bản class-wise của tác giả). Muốn tái lập đúng, tải `heyoujue/isic2018-classwise` và trỏ `CLASSWISE_DIR` tới đó.
- **Dữ liệu giả lập:** nếu không tải được dữ liệu thật (thiếu token hoặc mất mạng), notebook dùng ảnh tổng hợp để minh hoạ luồng chạy. Số liệu khi đó vô nghĩa về mặt y khoa.
- **Quick-infer:** mặc định transformer được fit một lần cho mỗi lớp. Đặt `ADAPT_EVERY_EPISODE=True` để fit lại mỗi episode (chậm hơn nhiều).
- **Ngẫu nhiên:** episode được sample ngẫu nhiên; seed cố định giúp tái lập trong một phiên, nhưng kết quả có thể lệch nếu đổi `SEED` hoặc số episode.
- **CRF:** `pydensecrf` không cài sẵn; post-processing mặc định tắt.
- **Bộ nhớ:** nếu GPU 4 GB bị OOM, giảm `IMG_SIZE` xuống 320 hoặc chạy CPU.

### Hướng mở rộng

- Thử `--nshot 5` để xem ảnh hưởng của số support.
- So sánh `--adapt-to first-episode` và `every-episode`.
- Thử `--postprocessing always` / `dynamic` sau khi cài `pydensecrf`.
- Tự tạo bản class-wise từ Task 3 (nếu có CSV nhãn chẩn đoán) để episode có ý nghĩa y khoa hơn.